# NB-05: adaLN-Zero Modulation

## Learning Objectives
- Understand why adaLN-Zero is used for timestep conditioning -- at different noise levels, different features matter, so the DiT needs a way to modulate its computation per-block based on the diffusion timestep
- Trace the six modulation parameters (shift, scale, gate for both attention and FFN) extracted via a single `.chunk(6)` operation from a learned per-block parameter
- See how GateModule implements gated residual connections, with the gate=0 identity property enabling stable training of deep networks
- Contrast Wan's small-random initialization (`torch.randn / dim**0.5`) with the DiT paper's zero initialization, and understand why both achieve near-identity behavior at init

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm and modulate function -- scale/shift mechanics), NB-02 (QKV projections), NB-03 (3D RoPE), NB-04 (self-attention and cross-attention)
- **Assumed concepts:** Residual connections, conditioning mechanisms (FiLM-style adaptive normalization), initialization strategies for deep networks

## Concept Map
- **adaLN-Zero modulation** -> used in every DiTBlock to condition self-attention and FFN branches on the timestep (NB-06, Track 2)
- **GateModule** -> used for gated residual connections in DiTBlock, enabling near-identity init (NB-06, Track 2)
- **modulate function** (NB-01) -> reused here for scale/shift application after normalization
- **Per-block learned offset** -> each of the DiT blocks has its own modulation parameter, allowing block specialization (NB-08, Track 2)

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import the symbols this notebook covers
from diffsynth.models.wan_video_dit import GateModule, DiTBlock, modulate, RMSNorm

import torch
import torch.nn as nn

print("Setup complete.")

## 1. The adaLN-Zero Mechanism

Adaptive Layer Norm (adaLN) conditions a neural network block on an external signal -- in WAN 2.2, that signal is the diffusion timestep embedding. The conditioning works by applying learned scale and shift to the normalized activations before the block's attention or FFN operations.

**Why this matters:** At different noise levels during the diffusion process, different features are important. Early denoising steps (high noise) need to establish global structure, while late steps (low noise) refine fine details. adaLN-Zero gives each DiTBlock a way to adjust its behavior based on which denoising step is being performed.

**adaLN-Zero** adds one more element beyond scale and shift: **gating**. Instead of directly adding the residual branch output, it multiplies it by a learned gate value:
```
output = x + gate * residual_branch(x)
```
When `gate = 0`, the output equals `x` exactly -- the block is an identity function. This is the **zero-init property**: if gates start at or near zero, each block starts training as a pass-through and gradually learns to incorporate its attention/FFN branches.

**How it works in practice:** Each DiTBlock has a `modulation` parameter of shape `[1, 6, dim]` that stores six vectors: (shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp). These are added to the time-embedding projection and then used to scale/shift normalized activations and gate residual connections.

Source: `diffsynth/models/wan_video_dit.py`, lines 190 and 212

## 2. GateModule: Gated Residual Connections

`GateModule` is the simplest module in the codebase:
```python
class GateModule(nn.Module):
    def forward(self, x, gate, residual):
        return x + gate * residual
```

The formula `x + gate * residual` gives three key behaviors:
- **gate = 0**: output = `x + 0 * residual = x` -- identity (the residual is completely suppressed, output equals input)
- **gate = 1**: output = `x + 1 * residual = x + residual` -- standard residual connection (full branch incorporated)
- **gate in (0, 1)**: output interpolates between identity and full residual

This enables the model to start near-identity at initialization and gradually learn to incorporate attention/FFN outputs during training.

Source: `diffsynth/models/wan_video_dit.py`, line 190

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 190 (GateModule)
B, S, dim = 1, 10, 384  # reduced config
gate_mod = GateModule()

x = torch.randn(B, S, dim)
residual = torch.randn(B, S, dim)

# gate=0: residual is suppressed -> output equals input
gate_zero = torch.zeros(B, 1, dim)
out_zero = gate_mod(x, gate_zero, residual)
assert out_zero.shape == torch.Size([B, S, dim])
assert torch.allclose(out_zero, x), "gate=0 should give identity"
print("gate=0: output == input (identity) -- training starts stable")
print(f"Max |output - input|: {(out_zero - x).abs().max().item()}")

# gate=1: residual is fully added
gate_one = torch.ones(B, 1, dim)
out_one = gate_mod(x, gate_one, residual)
assert out_one.shape == torch.Size([B, S, dim])
assert torch.allclose(out_one, x + residual), "gate=1 should add full residual"
print("gate=1: output == input + residual (full residual connection)")
print(f"Max |output - (x+residual)|: {(out_one - (x + residual)).abs().max().item()}")

## 3. Six Modulation Parameters

Each `DiTBlock` produces six modulation parameters by chunking a combined signal along the 6-parameter dimension:

```python
shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
    self.modulation + t_mod
).chunk(6, dim=1)
```

The six parameters break into two groups of three:

1. **shift_msa** -- additive shift for self-attention input (after norm1)
2. **scale_msa** -- multiplicative scale for self-attention input (after norm1)
3. **gate_msa** -- gate for self-attention residual
4. **shift_mlp** -- additive shift for FFN input (after norm2)
5. **scale_mlp** -- multiplicative scale for FFN input (after norm2)
6. **gate_mlp** -- gate for FFN residual

The `shift` and `scale` parameters feed into the `modulate` function from NB-01: `modulate(norm(x), shift, scale) = norm(x) * (1 + scale) + shift`. The `gate` parameters feed into the GateModule above.

Source: `diffsynth/models/wan_video_dit.py`, line 212 (modulation parameter), line 219 (six-chunk operation)

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 212, 219
dim = 384  # reduced config

# Per-block learned offset -- small random init (NOT zero-init like the original DiT paper)
modulation = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)  # [1, 6, dim]

# Simulate time-embedding projection (from WanModel.time_embedding)
t_mod = torch.randn(1, 6, dim)

# Combined: per-block learned offset + time-dependent signal
combined = modulation + t_mod
shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = combined.chunk(6, dim=1)

assert shift_msa.shape == torch.Size([1, 1, dim])
assert scale_msa.shape == torch.Size([1, 1, dim])
assert gate_msa.shape == torch.Size([1, 1, dim])
assert shift_mlp.shape == torch.Size([1, 1, dim])
assert scale_mlp.shape == torch.Size([1, 1, dim])
assert gate_mlp.shape == torch.Size([1, 1, dim])
print(f"Each parameter shape: {shift_msa.shape}")
print(f"Names: shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp")
print(f"\nSix modulation parameters (combined = modulation + t_mod):")
for name, val in zip(
    ["shift_msa", "scale_msa", "gate_msa", "shift_mlp", "scale_mlp", "gate_mlp"],
    [shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp]
):
    print(f"  {name}: shape={val.shape}, mean={val.mean():.6f}, std={val.std():.6f}")

## 4. Wan vs DiT Paper Initialization

This is an important distinction that is easy to get wrong:

| Aspect | Original DiT Paper (Peebles & Xie, 2023) | Wan Implementation |
|--------|------------------------------------------|--------------------|
| Gate init | **Zero** -- via zero-init of the final linear layer producing the conditioning signal | **Near-zero random** -- `torch.randn(1, 6, dim) / dim**0.5` |
| Block behavior at init | Exact identity (`gate = 0.0000...`) | Near-identity (gate has small random values) |
| Motivation | Guarantee stable initialization of all DiT blocks simultaneously | Allow each block a slightly different starting point, breaking symmetry |

**Both approaches achieve the key property:** at initialization, each block approximately preserves its input, enabling stable training of deep networks.

Wan's approach uses `torch.randn(1, 6, dim) / dim**0.5`, giving a standard deviation of approximately `1/sqrt(dim)`:
- With `dim=384` (reduced config): std is approximately 0.051
- With `dim=5120` (production S2V): std is approximately 0.014

The scaling by `1/sqrt(dim)` ensures that the initial modulation values are small regardless of the model dimension. Larger models get proportionally smaller initialization values, keeping the near-identity property.

Source: `diffsynth/models/wan_video_dit.py`, line 212
```python
self.modulation = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)
```

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 212

# Wan initialization: small random
dim = 384
wan_mod = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)
print(f"Wan init (dim={dim}):")
print(f"  Mean: {wan_mod.data.mean():.6f}")
print(f"  Std:  {wan_mod.data.std():.6f}  (expected ~{1/dim**0.5:.4f})")
print(f"  Max:  {wan_mod.data.abs().max():.6f}")

# DiT paper initialization: exact zero
dit_mod = nn.Parameter(torch.zeros(1, 6, dim))
print(f"\nDiT paper init:")
print(f"  Mean: {dit_mod.data.mean():.6f}")
print(f"  Std:  {dit_mod.data.std():.6f}")
print(f"  Max:  {dit_mod.data.abs().max():.6f}")

# Production config comparison
dim_prod = 5120
wan_mod_prod = nn.Parameter(torch.randn(1, 6, dim_prod) / dim_prod**0.5)
print(f"\nWan init (production dim={dim_prod}):")
print(f"  Std:  {wan_mod_prod.data.std():.6f}  (expected ~{1/dim_prod**0.5:.4f})")

# Verify both are near-zero
assert wan_mod.data.std() < 0.1, "Wan init std should be small"
assert dit_mod.data.std() == 0.0, "DiT init std should be exactly zero"
assert wan_mod_prod.data.std() < wan_mod.data.std(), "Larger dim -> smaller std"
print(f"\nBoth approaches give near-zero values at init (blocks start near-identity)")

## 5. Connecting Back to modulate

The extracted `shift_msa` and `scale_msa` parameters are used with the `modulate` function from NB-01. In the DiTBlock forward pass:
```python
input_x = modulate(self.norm1(x), shift_msa, scale_msa)
```

This applies the conditioning: first normalize `x` with `norm1` (a LayerNorm without learnable parameters), then scale by `(1 + scale_msa)` and shift by `shift_msa`. At initialization, `scale_msa` and `shift_msa` are small random values, so the output is close to the normalized input.

Note that `norm1` is a `LayerNorm` with `elementwise_affine=False` -- it has **no learnable parameters**. All the per-block conditioning comes from the `modulation` parameter via the `modulate` function.

Source: `diffsynth/models/wan_video_dit.py`, line 226

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 226
B, S, dim = 1, 10, 384
x = torch.randn(B, S, dim)
norm1 = nn.LayerNorm(dim, eps=1e-6, elementwise_affine=False)

# Re-extract modulation params with zero t_mod to see init behavior
mod_param = nn.Parameter(torch.randn(1, 6, dim) / dim**0.5)
t_mod_zero = torch.zeros(1, 6, dim)
s_msa, sc_msa, g_msa, s_mlp, sc_mlp, g_mlp = (mod_param + t_mod_zero).chunk(6, dim=1)

# Apply adaLN: normalize then modulate
normed = norm1(x)
input_x = modulate(normed, s_msa, sc_msa)
assert input_x.shape == torch.Size([B, S, dim])
print(f"x          -> {x.shape}")
print(f"norm1(x)   -> {normed.shape}")
print(f"modulate() -> {input_x.shape}")
print(f"Note: norm1 has elementwise_affine=False (no learnable params)")
print(f"\nDifference from normalized input: {(input_x - normed).abs().mean():.6f}")
print(f"(Small because scale_msa and shift_msa are near-zero at init)")

## 6. Per-Block Learned Offset

The `modulation` parameter is **unique to each DiTBlock instance**. Each block has its own `modulation` tensor of shape `[1, 6, dim]`, giving `6 * dim` learnable parameters per block for conditioning alone.

At inference time, the combined conditioning signal is:
```python
combined = self.modulation + t_mod  # per-block offset + global time projection
```

This allows each block to **specialize its response** to the timestep embedding. Early blocks might need different scale/shift/gate values than late blocks, and the learned offset lets them diverge naturally during training.

Source: `diffsynth/models/wan_video_dit.py`, line 212

In [ ]:
# Create two DiTBlocks with different modulation offsets
# Source: diffsynth/models/wan_video_dit.py, line 197
dim, num_heads, ffn_dim = 384, 4, 384 * 4
block1 = DiTBlock(has_image_input=True, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)
block2 = DiTBlock(has_image_input=True, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)

# Each block has its own modulation parameter (different random init)
print(f"Block 1 modulation std: {block1.modulation.data.std():.6f}")
print(f"Block 2 modulation std: {block2.modulation.data.std():.6f}")
print(f"Same values? {torch.allclose(block1.modulation, block2.modulation)}")
print(f"\nModulation shape: {block1.modulation.shape}  (1 x 6 x {dim})")
print(f"Total modulation params per block: {block1.modulation.numel():,}")

assert block1.modulation.shape == torch.Size([1, 6, dim])
assert block2.modulation.shape == torch.Size([1, 6, dim])
assert not torch.allclose(block1.modulation, block2.modulation), "Different blocks should have different inits"

print(f"\nWith dim={dim}, each block has {6 * dim:,} modulation parameters.")
print(f"The VACE encoder's VaceWanAttentionBlock uses the same modulation")
print(f"pattern with an additional control signal pathway -- covered in Track 4.")

## Summary

### Key Takeaways
- **adaLN-Zero** conditions each DiTBlock on the diffusion timestep by producing six modulation parameters per block: three for self-attention (shift_msa, scale_msa, gate_msa) and three for FFN (shift_mlp, scale_mlp, gate_mlp). This lets each block adjust its behavior based on the denoising step.
- **GateModule** implements `x + gate * residual`: gate=0 is identity (output == input), gate=1 is standard residual (output == input + branch_output). This property enables stable training by starting blocks as near-identity functions.
- **Six parameters** are extracted via `.chunk(6, dim=1)` from a combined signal of the per-block `modulation` offset plus the global time-embedding projection `t_mod`. Each parameter has shape `[1, 1, dim]`.
- **Wan's initialization** uses `torch.randn(1, 6, dim) / dim**0.5` (small random, std ~ 1/sqrt(dim)), NOT zero initialization as in the original DiT paper. Both approaches achieve near-identity block behavior at init, but Wan's breaks symmetry slightly.

### Source References
| Symbol | Location |
|--------|----------|
| GateModule | `diffsynth/models/wan_video_dit.py`, line 190 |
| DiTBlock | `diffsynth/models/wan_video_dit.py`, line 197 |
| DiTBlock.modulation | `diffsynth/models/wan_video_dit.py`, line 212 |
| modulate | `diffsynth/models/wan_video_dit.py`, line 64 |

## Exercises

### Exercise 1 -- Zero-init experiment
Replace the Wan modulation initialization with `torch.zeros(1, 6, dim)` in the six-parameter extraction cell. Verify that the gates become exactly zero. What does this mean for the residual path? Pass a zero gate into `GateModule` and confirm the output equals the input exactly.

### Exercise 2 -- Gate sweep
Create a GateModule and test gate values of 0.0, 0.5, and 1.0. For each gate value, compute the ratio `|output - x| / |residual|` (element-wise, then take the mean). Verify that the ratio equals the gate value. This confirms that GateModule linearly interpolates between identity and full residual.

### Exercise 3 -- Scaling experiment
Compare the modulation std for `dim=384`, `dim=1536`, and `dim=5120`. Verify that std scales as `1/sqrt(dim)` by computing `std * sqrt(dim)` for each -- the product should be approximately 1.0. Why might smaller init values matter for larger models? (Hint: think about the magnitude of the residual branch output in a deeper, wider network.)